In [28]:
from langchain_community.document_loaders import WebBaseLoader

In [29]:
from dotenv import load_dotenv
import os

load_dotenv()
pinecone_api_key = os.getenv("PINECONE_API_KEY")
pinecone_env = os.getenv("PINECONE_ENV")  # usually required too
groq_api_key = os.getenv("GROQ_API_KEY")
print(pinecone_api_key)
print(pinecone_env)

pcsk_4Z2cqB_6rDFy5tM3A6nCZZ5UnC4DUpk3kBZHhgX2LPz4mGBSS46EzCghjZais6d2Qjtqgt
us-east-1-aws


In [30]:
loader = WebBaseLoader("https://docs.langchain.com/oss/javascript/integrations/document_loaders/web_base_loader")
loader

In [31]:
docs = loader.load()
docs

[Document(metadata={'source': 'https://docs.langchain.com/oss/javascript/integrations/document_loaders/web_base_loader', 'language': 'No language found.'}, page_content='\n')]

In [32]:
from pinecone import Pinecone

pc = Pinecone(api_key=pinecone_api_key)
index = pc.Index("my-llm-index")

In [33]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=0)
chunks = text.split_documents(docs)

In [40]:
len(chunks)

0

In [34]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1922.35it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [36]:
from langchain_pinecone import PineconeVectorStore
vectorstore = PineconeVectorStore.from_documents(chunks, embeddings, index_name="my-llm-index", pinecone_api_key=pinecone_api_key)

In [39]:
query = "What is the purpose of the WebBaseLoader?"
result = vectorstore.similarity_search(query)
result[0].page_content

IndexError: list index out of range

In [ ]:
from langchain_groq import ChatGroq
chat = ChatGroq(model="gpt-4o", api_key=groq_api_key, temperature=0.5)

In [ ]:
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant that answers questions based on the following context: {context}"),
        ("human", "{question}")
    ]
)   

In [ ]:
langchain_core.documents import Document

documents_chain.invoke({
    "input" :"langsmith has two usage limits",
    "context" : "The LangSmith free tier has two usage limits: 1000 trace events per month and 100 trace events per minute. Trace events include all interactions with the LangSmith API, such as creating traces, adding spans, and retrieving trace data. If you exceed these limits, you may experience throttling or additional charges depending on your usage."
})

In [42]:
vectorstore._index.describe_index_stats()
# or use Pinecone client:
pc = Pinecone(api_key=pinecone_api_key)
print(pc.describe_index("my-llm-index"))

{'deletion_protection': 'disabled',
 'dimension': 1024,
 'embed': {'dimension': 1024,
           'field_map': {'text': 'text'},
           'metric': 'cosine',
           'model': 'llama-text-embed-v2',
           'read_parameters': {'dimension': 1024.0,
                               'input_type': 'query',
                               'truncate': 'END'},
           'vector_type': 'dense',
           'write_parameters': {'dimension': 1024.0,
                                'input_type': 'passage',
                                'truncate': 'END'}},
 'host': 'my-llm-index-izqnc9x.svc.aped-4627-b74a.pinecone.io',
 'metric': 'cosine',
 'name': 'my-llm-index',
 'spec': {'serverless': {'cloud': 'aws', 'region': 'us-east-1'}},
 'status': {'ready': True, 'state': 'Ready'},
 'tags': None,
 'vector_type': 'dense'}
